# MongoDB Aggregation Pipeline — Practical Analytics with PyMongo

## What you will learn in this course 🧐🧐

The **aggregation pipeline** is MongoDB’s built-in analytics engine: a sequence of **stages** that transform documents into the **answer** to a business question (think UNIX pipes for JSON). You’ll learn what each stage does, when to use it, and why it matters—then implement real analyses on `sample_analytics` using **only Python (PyMongo)**.

In this lecture, you will:

* understand the **goal** of the pipeline (answering questions server-side, at scale),
* learn each stage’s **purpose** (what / when / why / pitfalls),
* implement focused, minimal **code examples per stage**,
* apply performance habits (match early, project light, index wisely),
* know when to use **Aggregation vs Pandas** (and how they complement each other).

## Why aggregation if I already know Pandas?

The main reason is performance. Pandas is great for data analysis on relatively small datasets. Now let's say your DB holds 10GB of data, loading it all with Pandas will create latency both on your local machine (because you are dealing with 10GB of data in memory) and on the server (because you are pulling 10GB of data at once, multiplied by the number of analysts working in the organization). Aggregation pipelines are provide you with the following benefits:

* **Scale & memory:** Aggregations run on the **server**, use indexes, and stream—no need to load huge datasets into your laptop. Pandas is in-memory (great up to millions of rows, not hundreds of millions).
* **Network & cost:** `$match/$project` server-side ⇒ you ship **KB not GB** over the wire.
* **Latency for apps:** Indexed pipelines can answer **live** API/dashboard requests fast.
* **Governance:** Keep sensitive columns in DB; project only what’s allowed.
* **Document-native ops:** Arrays/nesting are first-class (`$unwind`, `$lookup`).
* **Workflow:** Do **reduction** in the DB (pipeline), then **rich analysis/viz** in Pandas on the small result.

Now does that mean that  you should  do Aggregation over Pandas? No. You will instead start with Aggregation to create small tidy slice of your dataset and then use Pandas to analyze the newly created subset.

## The aggregation pipeline flow

When creating an aggregation pipeline, there are **stages** that you will need to follow. Each stage takes documents in, transforms them, and passes them on. Here they are:

1. `$match` 👉 Filter rows (SQL WHERE)
2. `$project` 👉 Select/rename/compute columns
3. `$addFields` 👉 Add derived columns (without dropping others)
4. `$unwind` 👉 Explode arrays into rows
5. `$group` 👉 Aggregate (SUM/AVG/COUNT…) by key(s)
6. `$sort` / `$limit` 👉 Order & cap
7. `$lookup` 👉 Left join another collection
8. `$facet` 👉 Multiple aggregations in parallel

This stages reflect classic SQL queries. If that helps you better understand, here is a matching table:

| SQL concept                       | Aggregation stage               |
| --------------------------------- | ------------------------------- |
| `WHERE ...`                       | `$match`                        |
| `SELECT col, expr AS alias`       | `$project`, `$addFields`        |
| `UNNEST()` / explode arrays       | `$unwind`                       |
| `GROUP BY ... (SUM/AVG/COUNT)`    | `$group`                        |
| `ORDER BY ... LIMIT N`            | `$sort`, `$limit`               |
| `JOIN ... ON`                     | `$lookup` (left join semantics) |
| Multiple result sets in one query | `$facet`                        |


<Note type="important" title="Do I need to always follow these exact same stages?">

**No! Use only what you need**. A pipeline is a chain of *optional* stages, and you can repeat stages or change their order to fit the question. What matters is **order and shape**: some stages change the document shape (e.g., `$unwind`, `$group`), which affects what later stages can see. So don’t follow a fixed template. Compose the fewest, best-ordered stages that answer your question.

Here are some details as of when to use each stage:

| Stage              | When                                                                    | Why                                                        | Pitfalls                                                               |
| ------------------ | ----------------------------------------------------------------------- | ---------------------------------------------------------- | ---------------------------------------------------------------------- |
| `$match`           | Always, as early as possible (and again after `$unwind` for sub-fields) | Smaller inputs make later stages faster                    | Matching only **after** `$unwind` misses index use on top-level fields |
| `$project`         | Keep only fields you need; cast types; rename for clarity               | Reduces payload, improves readability, speeds later stages | Carrying large arrays/unused fields into `$group`/`$lookup`            |
| `$addFields`       | Attach a computed metric (e.g., signed cash) while preserving the doc   | Cleaner than stuffing expressions everywhere               | Over-computing early; compute only what you need                       |
| `$unwind`          | When aggregating by elements in an array (e.g., `transactions[]`)       | Makes nested data groupable                                | Row explosion; filter **before & after** to control volume             |
| `$group`           | When you need totals/averages/distinct counts per dimension             | This is where KPIs are produced                            | Grouping high-cardinality keys on huge inputs without pre-filters      |
| `$sort` / `$limit` | Top-N lists, stable presentation                                        | Makes output dashboard-friendly                            | Sorting large unfiltered sets; always reduce first                     |
| `$lookup`          | Enrich facts with dimensions (e.g., account limit/products)             | Complete analytics without leaving MongoDB                 | Joining on non-indexed keys → slow; index the foreign field            |
| `$facet`           | Produce several tiles (top-N, KPI, trend) from the same scan            | Efficiency and consistent snapshot                         | Returning huge nested payloads; keep each facet tight                  |


### Practical rules of thumb

* **Use only what’s needed.** Many pipelines are just `$match → $group → $sort → $limit`. You don’t *have* to `$project` or `$unwind` if you don’t need them.
* **Order matters.** Filter early (`$match`), then reduce/compute (`$group`), then present (`$sort/$limit/$project`).
* **Repeat as needed.** It’s common to `$match` twice (before and after `$unwind`).
* **Stage constraints to remember:**

  * `$out` / `$merge` **must be last**.
  * `$geoNear` **must be first** (except when using Atlas `$search`, which must be first in that case).
  * `$sort` before `$limit` for “Top-N”.
  * After `$group`, only the accumulator outputs (and `_id`) remain—fields not in `_id` are gone unless you re-add/lookup them.
* **Performance:** early `$match` and lean `$project` shrink data; indexes help pre-unwind filters and join keys; avoid unnecessary `$unwind` (it explodes rows).


</Note>

## Demo 

Let's see examples for each stage of the aggregation pipeline. We will use our sample dataset `sample_analytics` again

In [ ]:
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

USERNAME = "xxxxxx_db_user" # Replace with your username 
PASSWORD = "xxxxxx_db_password" # Replace with your password
CLUSTER_NAME= "Cluster0" # Replace with your cluster name
MONGODB_URI=f"mongodb+srv://{USERNAME}:{PASSWORD}@{CLUSTER_NAME.lower()}.1tsvfmh.mongodb.net/?retryWrites=true&w=majority&appName={CLUSTER_NAME.lower()}"

client = MongoClient(MONGODB_URI, server_api=ServerApi('1'))

try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

db = client["sample_analytics"]
customers = db.customers
accounts = db.accounts
transactions = db.transactions

print(customers.estimated_document_count()) 

Pinged your deployment. You successfully connected to MongoDB!
503


## Stage 1 - `$match` (Filter rows (SQL `WHERE`))

In [3]:
# Keep only documents for a specific account_id (top-level, index-friendly)
pipeline = [
    {"$match": {"account_id": 716662}}
]
list(transactions.aggregate(pipeline))[:2]  # peek a few

[{'_id': ObjectId('5ca4bbc1a2dd94ee58161cb2'),
  'account_id': 716662,
  'transaction_count': 48,
  'bucket_start_date': datetime.datetime(1962, 5, 13, 0, 0),
  'bucket_end_date': datetime.datetime(2016, 12, 27, 0, 0),
  'transactions': [{'date': datetime.datetime(2008, 3, 19, 0, 0),
    'amount': 8592,
    'transaction_code': 'buy',
    'symbol': 'amd',
    'price': '6.25868566899633460565155473886989057064056396484375',
    'total': '53774.62726801650693175815832'},
   {'date': datetime.datetime(2008, 5, 27, 0, 0),
    'amount': 5360,
    'transaction_code': 'sell',
    'symbol': 'amd',
    'price': '6.8846943545406844577883020974695682525634765625',
    'total': '36901.96174033806869374529924'},
   {'date': datetime.datetime(2015, 11, 5, 0, 0),
    'amount': 4878,
    'transaction_code': 'buy',
    'symbol': 'amd',
    'price': '2.226658409790678749828884974704124033451080322265625',
    'total': '10861.63972295893094166530091'},
   {'date': datetime.datetime(2015, 10, 21, 0, 0),
  

## Stage 2 - `$project` (**Select/rename/cast** columns)

In [13]:
# Keep just symbol, amount, and price (cast to number) from each sub-transaction
pipeline = [
    {"$match": {"account_id": 716662}}, 
    {"$project": {
        "_id": 0,
        "symbol": "$transactions.symbol",
        "amount": "$transactions.amount",
        "price": {
            "$convert": {
                "input": "$transactions.price",
                "to": "double",
                "onError": None,
                "onNull": None
            } # sample stores price as string so we convert that to float
    }}}
]
list(transactions.aggregate(pipeline))[:2]  # peek a few

[{'symbol': ['amd',
   'amd',
   'amd',
   'znga',
   'znga',
   'amd',
   'ibm',
   'ibm',
   'amd',
   'znga',
   'ibm',
   'znga',
   'ibm',
   'ibm',
   'ibm',
   'amd',
   'amd',
   'ibm',
   'amd',
   'amd',
   'amd',
   'amd',
   'ibm',
   'znga',
   'znga',
   'ibm',
   'amd',
   'ibm',
   'amd',
   'amd',
   'ibm',
   'ibm',
   'amd',
   'znga',
   'amd',
   'amd',
   'ibm',
   'amd',
   'ibm',
   'ibm',
   'znga',
   'ibm',
   'amd',
   'ibm',
   'znga',
   'amd',
   'znga',
   'amd'],
  'amount': [8592,
   5360,
   4878,
   653,
   8043,
   5675,
   7127,
   9428,
   4762,
   3750,
   7508,
   2157,
   6178,
   2348,
   1636,
   4041,
   8320,
   5839,
   7424,
   1570,
   4035,
   2396,
   7561,
   8861,
   989,
   6991,
   4448,
   1390,
   5871,
   5780,
   9500,
   242,
   1810,
   3630,
   6707,
   9063,
   4984,
   2859,
   3420,
   346,
   5663,
   3840,
   5448,
   5852,
   9019,
   8260,
   2753,
   3875],
  'price': None}]

<Note type="note" title="$project is short for projection">

Don't get confused, `$project` is short for **projection** meaning selecting which column you want!

</Note>

## Stage 3 - `$addFields` — **Add derived columns** (without dropping others)

In [15]:
# Signed cash: buy = negative total, sell = positive total
pipeline = [
    {"$unwind": "$transactions"},
    {"$project": {
        "account_id": 1,
        "code": "$transactions.transaction_code",
        "total": {"$toDouble": "$transactions.total"}
    }},
    {"$addFields": {
        "signedCash": {
            "$cond": [{"$eq": ["$code", "buy"]}, {"$multiply": [-1, "$total"]}, "$total"]
        }
    }}
]
list(transactions.aggregate(pipeline))[:2]

[{'_id': ObjectId('5ca4bbc1a2dd94ee58161cb1'),
  'account_id': 443178,
  'code': 'buy',
  'total': 143572.10391126573,
  'signedCash': -143572.10391126573},
 {'_id': ObjectId('5ca4bbc1a2dd94ee58161cb1'),
  'account_id': 443178,
  'code': 'buy',
  'total': 223169.68432630083,
  'signedCash': -223169.68432630083}]

## Stage 4 - `$unwind` ( **Explode arrays into rows** (SQL `UNNEST`))

This stage can be a bit trickier. Let's explain what it does in more details. What we mean by *Exploding arrays into documents* is that if a document contains an array (i.e a list) of elements like this:

```json
{
  "account_id": 716662,
  "transactions": [
    { "symbol": "amd", "amount": 8592, "price": "6.25", "transaction_code": "buy" },
    { "symbol": "ibm", "amount": 242,  "price": "164.97", "transaction_code": "sell" }
  ]
}
```

After applying `$unwind` to `transactions` it will look like this:

```json
{ "account_id": 716662, "transactions": { "symbol": "amd", "amount": 8592, ... } }
{ "account_id": 716662, "transactions": { "symbol": "ibm", "amount": 242,  ... } }
```

This is especially useful when you have deeply nested elements that you want to `$group` by a given attribute (inside that element). Instead of looking at an example here, let's directly go to the next stage to understand better

<Note type="tip">

If your goal is to keep only the matching elements inside each original document (not to aggregate across docs), `$filter` avoids row explosion:

```python
pipeline = [
    {"$project": {
        "_id": 0,
        "account_id": 1,
        "transactions": {
            "$filter": {
                "input": "$transactions",
                "as": "t",
                "cond": {
                    "$and": [
                        {"$eq": ["$$t.symbol", "amd"]},
                        {"$eq": ["$$t.transaction_code", "buy"]},
                        {"$gt": ["$$t.amount", 8000]}
                    ]
                }
            }
        }
    }}
]
list(db.transactions.aggregate(pipeline))[:2]
```


</Note>

## Stage 5 - `$group` ( **Aggregate** (SUM/AVG/COUNT…) by key(s))

In [17]:
# Total traded volume per symbol
pipeline = [
    {"$unwind": "$transactions"},
    {"$group": {
        "_id": "$transactions.symbol",
        "volume": {"$sum": "$transactions.amount"},
        "trades": {"$sum": 1}
    }},
    {"$project": {"_id": 0, "symbol": "$_id", "volume": 1, "trades": 1}}
]
list(transactions.aggregate(pipeline))[:2]

[{'volume': 26108705, 'trades': 5263, 'symbol': 'nvda'},
 {'volume': 23506255, 'trades': 4672, 'symbol': 'nflx'}]

## Stage 6 - `$sort` and `$limit` (**Order & cap** results)

In [18]:
# Top 10 symbols by volume
pipeline = [
    {"$unwind": "$transactions"},
    {"$group": {"_id": "$transactions.symbol", "volume": {"$sum": "$transactions.amount"}}},
    {"$sort": {"volume": -1}},
    {"$limit": 10},
    {"$project": {"_id": 0, "symbol": "$_id", "volume": 1}}
]
list(transactions.aggregate(pipeline))

[{'volume': 27463715, 'symbol': 'adbe'},
 {'volume': 27232371, 'symbol': 'ebay'},
 {'volume': 27099929, 'symbol': 'crm'},
 {'volume': 27029894, 'symbol': 'goog'},
 {'volume': 26108705, 'symbol': 'nvda'},
 {'volume': 25511429, 'symbol': 'amzn'},
 {'volume': 25074207, 'symbol': 'amd'},
 {'volume': 24831259, 'symbol': 'ibm'},
 {'volume': 24632151, 'symbol': 'aapl'},
 {'volume': 24233156, 'symbol': 'csco'}]

## Stage 7 - `$lookup` (Left join with another collection)

If you need to add more information from another collection to your document. You can definitely do so:

In [19]:
# Join per-account AMD volume with account limit & products
pipeline = [
    {"$unwind": "$transactions"},
    {"$match": {"transactions.symbol": "amd"}},
    {"$group": {"_id": "$account_id", "amd_volume": {"$sum": "$transactions.amount"}}},
    {"$lookup": {
        "from": "accounts",
        "localField": "_id",
        "foreignField": "account_id",
        "as": "account"
    }},
    {"$unwind": "$account"},
    {"$project": {
        "_id": 0,
        "account_id": "$_id",
        "amd_volume": 1,
        "limit": "$account.limit",
        "products": "$account.products"
    }}
]
list(transactions.aggregate(pipeline))[:3]

[{'amd_volume': 46791,
  'account_id': 929644,
  'limit': 10000,
  'products': ['Brokerage', 'InvestmentStock', 'Derivatives', 'Commodity']},
 {'amd_volume': 65089,
  'account_id': 134434,
  'limit': 10000,
  'products': ['InvestmentStock', 'InvestmentFund', 'Commodity', 'Brokerage']},
 {'amd_volume': 89336,
  'account_id': 467651,
  'limit': 10000,
  'products': ['Commodity', 'Brokerage', 'Derivatives', 'InvestmentStock']}]

<Note type="note" title="This is a left join">

Meaning you get all the info from your collection and if there is a match with the foreign collection, you will see the data. If there is no match, the data from the "left" collection will still be output.

</Note>

## Stage 8 - `$facet` (**Run multiple aggregations in parallel**)

If you need to run multiple aggregations, you need to use `$facet` like so:

In [20]:
from datetime import datetime, timezone, timedelta

now = datetime.now(timezone.utc)
pipeline = [
    {"$unwind": "$transactions"},
    {"$project": {
        "symbol": "$transactions.symbol",
        "date": "$transactions.date",
        "price": {"$toDouble": "$transactions.price"}
    }},
    {"$facet": {
        "top_symbols": [
            {"$group": {"_id": "$symbol", "trades": {"$sum": 1}}},
            {"$sort": {"trades": -1}}, {"$limit": 5},
            {"$project": {"_id": 0, "symbol": "$_id", "trades": 1}}
        ],
        "avg_price_per_symbol": [
            {"$group": {"_id": "$symbol", "avg_price": {"$avg": "$price"}}},
            {"$project": {"_id": 0, "symbol": "$_id", "avg_price": 1}}
        ],
        "trades_last_30d": [
            {"$match": {"date": {"$gte": now - timedelta(days=30)}}},
            {"$count": "count"}
        ]
    }}
]
list(transactions.aggregate(pipeline))

[{'top_symbols': [{'trades': 5559, 'symbol': 'ebay'},
   {'trades': 5500, 'symbol': 'goog'},
   {'trades': 5498, 'symbol': 'adbe'},
   {'trades': 5419, 'symbol': 'crm'},
   {'trades': 5263, 'symbol': 'nvda'}],
  'avg_price_per_symbol': [{'avg_price': 10.355889801690946, 'symbol': 'amd'},
   {'avg_price': 63.08828526322907, 'symbol': 'ibm'},
   {'avg_price': 16.252898007438652, 'symbol': 'ebay'},
   {'avg_price': 202.99011105801924, 'symbol': 'amzn'},
   {'avg_price': 42.00711379439306, 'symbol': 'crm'},
   {'avg_price': 82.38210639503083, 'symbol': 'fb'},
   {'avg_price': 3.3834961907851064, 'symbol': 'znga'},
   {'avg_price': 61.19512590107342, 'symbol': 'sap'},
   {'avg_price': 18.374236423132636, 'symbol': 'nvda'},
   {'avg_price': 22.816737273446503, 'symbol': 'msft'},
   {'avg_price': 42.17860240708068, 'symbol': 'nflx'},
   {'avg_price': 17.93275484225655, 'symbol': 'csco'},
   {'avg_price': 669.7785706132923, 'symbol': 'goog'},
   {'avg_price': 26.07097204985134, 'symbol': 'team

## Resources 📚📚

* MongoDB Docs — Aggregation Pipeline: [https://www.mongodb.com/docs/manual/aggregation/](https://www.mongodb.com/docs/manual/aggregation/)
* Operators — Expression, Accumulator, Array: [https://www.mongodb.com/docs/manual/reference/operator/](https://www.mongodb.com/docs/manual/reference/operator/)
* PyMongo — `Collection.aggregate`: [https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.aggregate](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.aggregate)
* Data Modeling Best Practices: [https://www.mongodb.com/developer/products/mongodb/schema-design-best-practices/](https://www.mongodb.com/developer/products/mongodb/schema-design-best-practices/)
* Indexes & Explain Plans: [https://www.mongodb.com/docs/manual/indexes/](https://www.mongodb.com/docs/manual/indexes/)
* Atlas Sample Datasets: [https://www.mongodb.com/docs/atlas/sample-data/](https://www.mongodb.com/docs/atlas/sample-data/)